In [4]:


import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104], # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38], # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95], # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]  # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]



## Task 1: Cleaning Machakos Rainfall Data
**Approach:** I will write a cleaning function that iterates through `machakos_raw`, stripping whitespace and using a `try...except ValueError` block to parse valid numbers. Any invalid or missing values will be caught, logged, and imputed using the rounded average of the valid entries.

In [5]:
def clean_machakos(raw_data, month_names):
    """
    Cleans raw rainfall data strings, handles invalid values via try/except,
    and imputes missing entries with the rounded average of valid readings.
    """
    valid_values = []
    parsed_data = []

    # First pass: try converting valid numbers, track invalid ones
    for item in raw_data:
        cleaned_item = item.strip()
        try:
            val = float(cleaned_item)
            parsed_data.append(val)
            valid_values.append(val)
        except (ValueError, TypeError):
            parsed_data.append(None)

    # Calculate rounded average of valid numbers for imputation
    avg_val = round(sum(valid_values) / len(valid_values))

    final_machakos = []
    imputed_months = []

    # Second pass: impute missing/invalid values and track which months were fixed
    for i, val in enumerate(parsed_data):
        if val is None:
            final_machakos.append(avg_val)
            imputed_months.append(month_names[i])
        else:
            final_machakos.append(int(val))

    print(f"Imputed months (replaced with average {avg_val}): {imputed_months}")
    return final_machakos

machakos_clean = clean_machakos(machakos_raw, months)

Imputed months (replaced with average 68): ['Apr', 'Sep']



## Task 2: NumPy Array Construction
**Approach:** I will combine all five county lists into a single 2D NumPy array, then verify its shape and data type.

In [6]:
# Combine clean rows with the cleaned Machakos list
all_rainfall = clean_rows + [machakos_clean]

# Create the 5 x 12 NumPy array
rainfall_array = np.array(all_rainfall)

print("Array Shape:", rainfall_array.shape)
print("Array Data Type:", rainfall_array.dtype)

Array Shape: (5, 12)
Array Data Type: int64


## Task 3: Calculations of Totals and Averages
**Approach:** I will use NumPy axis operations (`axis=1` for annual totals per county and `axis=0` for monthly averages across all counties) to perform vector calculations without loops.

In [7]:
# Annual total rainfall per county (sum across columns, axis=1)
annual_totals = np.sum(rainfall_array, axis=1)

# Average rainfall per month across all counties (mean down rows, axis=0)
monthly_averages = np.mean(rainfall_array, axis=0)

for county, total in zip(counties, annual_totals):
    print(f"{county} Annual Total: {total} mm")

print("\nMonthly Averages Across All Counties:")
for m, avg in zip(months, monthly_averages):
    print(f"{m}: {avg:.2f} mm")

Nairobi Annual Total: 1045 mm
Nakuru Annual Total: 916 mm
Kisumu Annual Total: 1371 mm
Mombasa Annual Total: 1075 mm
Machakos Annual Total: 811 mm

Monthly Averages Across All Counties:
Jan: 47.00 mm
Feb: 47.60 mm
Mar: 91.00 mm
Apr: 164.40 mm
May: 171.20 mm
Jun: 67.60 mm
Jul: 56.60 mm
Aug: 58.20 mm
Sep: 64.20 mm
Oct: 67.80 mm
Nov: 127.40 mm
Dec: 80.60 mm


## Task 4: Wettest Month Identification
**Approach:** I will use `np.argmax(rainfall_array, axis=1)` to find the index of the highest rainfall value for each row, and map that index back to the corresponding month name using `zip()`.

In [8]:
# Find index of max value for each row
wettest_month_indices = np.argmax(rainfall_array, axis=1)

wettest_months = []
for county, idx in zip(counties, wettest_month_indices):
    month_name = months[idx]
    wettest_months.append(month_name)
    print(f"{county}'s wettest month: {month_name} ({rainfall_array[counties.index(county), idx]} mm)")


Nairobi's wettest month: Apr (219 mm)
Nakuru's wettest month: Apr (145 mm)
Kisumu's wettest month: Apr (240 mm)
Mombasa's wettest month: May (240 mm)
Machakos's wettest month: Nov (182 mm)


## Task 5: Dry-Month Counts
**Approach:** I will create a boolean mask where rainfall is less than 50 mm, and sum across the rows to count dry months per county.

In [9]:
# Boolean mask for months below 50 mm
dry_mask = rainfall_array < 50

# Sum boolean values (True=1, False=0) across columns (axis=1)
dry_month_counts = np.sum(dry_mask, axis=1)

for county, count in zip(counties, dry_month_counts):
    print(f"{county} dry months (<50mm): {count}")

Nairobi dry months (<50mm): 4
Nakuru dry months (<50mm): 3
Kisumu dry months (<50mm): 0
Mombasa dry months (<50mm): 2
Machakos dry months (<50mm): 6


## Task 6: Planting Window Analysis
**Approach:** I will use the shifted-array technique `R[:, :-1] + R[:, 1:]` to find the two consecutive months with the highest combined rainfall for each county.

In [10]:
# Shifted array technique: add month i and month i+1 across all rows
two_month_rolling = rainfall_array[:, :-1] + rainfall_array[:, 1:]

# Find the index of the max 2-month window for each county
best_window_indices = np.argmax(two_month_rolling, axis=1)

planting_windows = []
for county, idx in zip(counties, best_window_indices):
    m1 = months[idx]
    m2 = months[idx + 1]
    window_str = f"{m1}-{m2}"
    planting_windows.append(window_str)
    print(f"Recommended planting window for {county}: {window_str}")

Recommended planting window for Nairobi: Apr-May
Recommended planting window for Nakuru: Apr-May
Recommended planting window for Kisumu: Apr-May
Recommended planting window for Mombasa: Apr-May
Recommended planting window for Machakos: Nov-Dec


## Task 7: Generate CSV Summary Report
**Approach:** I will use Python's built-in `csv` module to write the processed metrics out to `rainfall_summary.csv`.

In [11]:
import csv

# Write summary data to rainfall_summary.csv

csv_filename = "rainfall_summary.csv"

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # Write header
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])

    # Write rows for each county
    for i, county in enumerate(counties):
        writer.writerow([
            county,
            int(annual_totals[i]),
            wettest_months[i],
            int(dry_month_counts[i]),
            planting_windows[i]
        ])

print(f"Successfully generated {csv_filename}!")

Successfully generated rainfall_summary.csv!


## Planting Advisory

Based on our multi-county NumPy rainfall analysis, we advise the NGO to tailor its SMS planting window alerts dynamically to match each region's peak consecutive rainfall periods. Specifically, farmers across all five counties should be guided toward their optimal wet-season windows while monitoring recurring dry-month constraints. However, field teams must note an important data-quality caveat regarding Machakos County, where raw text inputs contained missing and malformed entries that required statistical imputation. Because those imputed monthly values rely on a rounded average of valid readings rather than physical weather-station logs, advisories for Machakos should be interpreted with cautious operational flexibility. Implementing these targeted, data-backed SMS campaigns will help maximize agricultural yields while keeping local farmers informed of underlying data limitations.